# 🪨 Multi-Method Shale Volume (Vsh) Petrophysical Evaluation
### Well: OMA-1321 | Field: Omar, Syria (Al Furat Petroleum Company)
**Target Role:** Baker Hughes Wireline Field Engineering Internship

---
### Project Workflow:
1. **Data Ingestion & Cleaning:** Read raw open-hole LAS 2.0 log data and handle null values (`-999.25`).
2. **Calculate 5 Shale Volume Methods:**
   - Method 1: Linear Gamma Ray Index
   - Method 2: Larionov Tertiary (for young/unconsolidated sands)
   - Method 3: Larionov Older (consolidated rocks)
   - Method 4: Stieber (1970) (Middle East clastic standard)
   - Method 5: Density-Neutron combination (GR-independent)
3. **Spectral Gamma Ray QC (Uranium Analysis):** Calculate $U = SGR - CGR$ to identify organic-rich source rocks.
4. **Petrophysical Audit & Validation:** Compare all 5 methods against pre-stored LAS `VSH` (RMSE, Bias, R).
5. **Export:** Save cleaned curves and validation tables to Excel.

In [ ]:
import os
import lasio
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Load LAS file
las_path = os.path.join('data', 'oma1321.las')
las = lasio.read(las_path)
df = las.df().reset_index()
df.replace(-999.25, np.nan, inplace=True)
df.columns = [c.strip().upper() for c in df.columns]

# Clean depth column
if 'DEPT' in df.columns and 'DEPTH' in df.columns:
    df.drop(columns=['DEPT'], inplace=True)
elif 'DEPT' in df.columns:
    df.rename(columns={'DEPT': 'DEPTH'}, inplace=True)

# Filter to valid logged interval
df_clean = df[df['CGR'].notna()].copy().reset_index(drop=True)
print(f'Successfully loaded OMA-1321 with {len(df_clean):,} valid depth samples.')
print(f'Depth Range: {df_clean["DEPTH"].min():.1f} m to {df_clean["DEPTH"].max():.1f} m MD')

## 1. Shale Volume Calculation (5 Methods)
$$
I_{GR} = \frac{CGR - CGR_{clean}}{CGR_{shale} - CGR_{clean}}
$$
- **Larionov Tertiary (1969):** $V_{sh} = 0.083 \times (2^{3.7 \times I_{GR}} - 1)$
- **Larionov Older (1969):** $V_{sh} = 0.33 \times (2^{2 \times I_{GR}} - 1)$
- **Stieber (1970):** $V_{sh} = \frac{I_{GR}}{3 - 2 \times I_{GR}}$
- **Density-Neutron:** $V_{sh} = \frac{\phi_N - \phi_D}{\phi_{N,sh} - \phi_{D,sh}}$

In [ ]:
# Gamma Ray Clean (P05) and Shale (P95) baselines
gr_clean = float(df_clean['CGR'].quantile(0.05))
gr_shale = float(df_clean['CGR'].quantile(0.95))

# 1. Linear Gamma Ray Index
igr = ((df_clean['CGR'] - gr_clean) / (gr_shale - gr_clean)).clip(0.0, 1.0)
df_clean['VSH_LINEAR'] = igr

# 2. Larionov Tertiary (Young Rocks)
df_clean['VSH_LAR_TERTIARY'] = (0.083 * (2.0 ** (3.7 * igr) - 1.0)).clip(0.0, 1.0)

# 3. Larionov Older (Consolidated Rocks)
df_clean['VSH_LAR_OLDER'] = (0.33 * (2.0 ** (2.0 * igr) - 1.0)).clip(0.0, 1.0)

# 4. Stieber (1970)
df_clean['VSH_STIEBER'] = (igr / (3.0 - 2.0 * igr).clip(lower=1e-6)).clip(0.0, 1.0)

# 5. Density-Neutron Method (Independent of GR)
rho_ma = 2.65
rho_fl = 1.00
phid = ((rho_ma - df_clean['DENS']) / (rho_ma - rho_fl)).clip(0.0, 0.60)
df_clean['PHID'] = phid
df_clean['VSH_DENSITY_NEUTRON'] = ((df_clean['CNLF'] - phid) / (0.45 - 0.05)).clip(0.0, 1.0)

# Bonus: Uranium extraction
df_clean['URANIUM'] = (df_clean['SGR'] - df_clean['CGR']).clip(lower=0.0)
df_clean['ORGANIC_FLAG'] = np.where(df_clean['URANIUM'] >= 8.0, 1, 0)

print('All 5 methods calculated successfully!')

## 2. Validation & Petrophysical Audit vs Stored VSH
Quantifying how closely each method matches the original pre-interpreted `VSH` in the LAS file.

In [ ]:
methods = ['VSH_LINEAR', 'VSH_LAR_TERTIARY', 'VSH_LAR_OLDER', 'VSH_STIEBER', 'VSH_DENSITY_NEUTRON']
stored = df_clean['VSH']
val_rows = []

for m in methods:
    calc = df_clean[m]
    mask = calc.notna() & stored.notna()
    diff = calc[mask] - stored[mask]
    rmse = np.sqrt((diff ** 2).mean())
    bias = diff.mean()
    corr = np.corrcoef(calc[mask], stored[mask])[0, 1]
    val_rows.append({
        'Method': m,
        'RMSE': round(float(rmse), 4),
        'Bias': round(float(bias), 4),
        'Correlation (R)': round(float(corr), 4)
    })

df_val = pd.DataFrame(val_rows)
df_val

## 3. Wireline Multi-Track Plot (Matplotlib)

In [ ]:
# Depth slice for visualization
sub = df_clean[(df_clean['DEPTH'] >= 2100) & (df_clean['DEPTH'] <= 2350)]

fig, axes = plt.subplots(1, 4, figsize=(14, 10), sharey=True)
depth = sub['DEPTH']

# Track 1: CGR vs SGR
axes[0].plot(sub['CGR'], depth, color='green', lw=1.5, label='CGR (Th+K)')
axes[0].plot(sub['SGR'], depth, color='olive', ls='--', lw=1.2, label='SGR (Total)')
axes[0].set_xlim(0, 150)
axes[0].set_xlabel('GR (GAPI)', color='green', fontweight='bold')
axes[0].set_ylabel('Depth (m MD)', fontweight='bold')
axes[0].invert_yaxis()
axes[0].grid(True, ls=':')
axes[0].legend(loc='upper right')

# Track 2: Density - Neutron
axes[1].plot(sub['DENS'], depth, color='red', lw=1.5, label='DENS (g/cm³)')
axes[1].set_xlim(1.95, 2.95)
axes[1].set_xlabel('DENS (g/cm³)', color='red', fontweight='bold')
axes[1].grid(True, ls=':')

ax2_nphi = axes[1].twiny()
ax2_nphi.plot(sub['CNLF'], depth, color='blue', ls='-.', lw=1.5, label='CNLF (v/v)')
ax2_nphi.set_xlim(0.45, -0.15)  # standard reverse scale
ax2_nphi.set_xlabel('CNLF (v/v)', color='blue', fontweight='bold')

# Track 3: All Vsh Methods
axes[2].plot(sub['VSH_LINEAR'], depth, color='red', ls=':', lw=1.2, label='Linear')
axes[2].plot(sub['VSH_LAR_TERTIARY'], depth, color='green', lw=1.8, label='Larionov Tertiary')
axes[2].plot(sub['VSH_STIEBER'], depth, color='purple', ls='--', lw=1.5, label='Stieber')
axes[2].plot(sub['VSH_DENSITY_NEUTRON'], depth, color='blue', ls='-.', lw=1.5, label='Dens-Neut')
axes[2].set_xlim(0, 1.0)
axes[2].set_xlabel('Calculated Vsh (v/v)', fontweight='bold')
axes[2].grid(True, ls=':')
axes[2].legend(loc='lower right', fontsize=8)

# Track 4: Best Method vs Stored VSH
axes[3].plot(sub['VSH_LAR_TERTIARY'], depth, color='green', lw=2.0, label='Larionov T (Calc)')
axes[3].plot(sub['VSH'], depth, color='black', ls='--', lw=1.5, label='Stored VSH (LAS)')
axes[3].fill_betweenx(depth, 0, sub['VSH_LAR_TERTIARY'], color='gold', alpha=0.3, label='Reservoir (<0.35)')
axes[3].set_xlim(0, 1.0)
axes[3].set_xlabel('Validation Track', fontweight='bold')
axes[3].grid(True, ls=':')
axes[3].legend(loc='lower right', fontsize=8)

plt.suptitle('Well OMA-1321 — Shale Volume Multi-Method Evaluation', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Export Cleaned Data to Excel
The cleaned and calculated petrophysical results are saved to `OMA1321_Shale_Volume_Evaluation.xlsx`.

In [ ]:
excel_out = 'OMA1321_Shale_Volume_Evaluation.xlsx'
with pd.ExcelWriter(excel_out, engine='openpyxl') as writer:
    df_clean.to_excel(writer, sheet_name='Cleaned_Petrophysics', index=False)
    df_val.to_excel(writer, sheet_name='Validation_Summary', index=False)
print(f'Exported cleaned dataset and validation summary to: {excel_out}')